In [1]:
import pandas as pd
import numpy as np
import joblib
import time
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from functions import feature_engineering, label, create_sequences, import_csv

In [2]:
#import model and belonging objects
model_dir = Path("models/saved_lstm_autoencoder")

model = tf.keras.models.load_model(model_dir / "lstm_autoencoder.keras")

scaler = joblib.load(model_dir / "scaler.pkl")
threshold = joblib.load(model_dir / "threshold.pkl")
feature_columns = joblib.load(model_dir / "feature_columns.pkl")
window_size = joblib.load(model_dir / "window_size.pkl")

print("Modellen er lastet inn.")
print("Threshold:", threshold)
print("Window size:", window_size)
print("Antall features:", len(feature_columns))

Modellen er lastet inn.
Threshold: 0.10404626509857504
Window size: 30
Antall features: 18


In [3]:
#Define CSV
benign_csv = [
    "Benign-14.csv",
    "Benign-15.csv",
    #"Benign-16.csv",
    #"Benign-17.csv",
    #"Benign-18.csv",
    #"Benign-19.csv",
    #"Benign-20.csv",
    #"Benign-21.csv",
    #"Benign-22.csv",
    #"Benign-23.csv",
    #"Benign-24.csv",
    #"Benign-25.csv",
    #"Benign-26.csv",
    #"Benign-27.csv",
    #"Benign-28.csv",
    #"Benign-29.csv",
    #"Benign-30.csv",
    #"Benign-31.csv",
    #"Benign-32.csv"
]

csv_folder = Path.cwd()/"CSV files"

In [4]:
#import csv
benign_df = import_csv(benign_csv, csv_folder)
print("benign_df shape:", benign_df.shape)

Reading: C:\Users\Even_\OneDrive\Master\Dataset\Jupyter\CSV files\Benign-14.csv
Reading: C:\Users\Even_\OneDrive\Master\Dataset\Jupyter\CSV files\Benign-15.csv
benign_df shape: (430829, 14)


In [5]:
#feature enginering
train = feature_engineering(benign_df)
feature_columns = train.columns.tolist()
print("train shape:", train.shape)

train shape: (430829, 18)


In [6]:
#Scaling
X_train_scaled = scaler.transform(train)

In [7]:
#Sequence data
X_seq = create_sequences(X_train_scaled,window_size)

In [8]:
#Get latent layer
encoder_model = Model(
    inputs=model.input,
    outputs=model.get_layer("dense").output
)

X_train_latent = encoder_model.predict(X_seq)
print("X_train_latent shape:", X_train_latent.shape)

13463/13463 ━━━━━━━━━━━━━━━━━━━━ 103s 8ms/step
X_train_latent shape: (430800, 16)


In [9]:
X_train_latent

array([[1.2212557 , 0.        , 0.485483  , ..., 0.        , 0.        ,
        2.4043345 ],
       [0.32330287, 0.27126035, 0.8433764 , ..., 0.78155273, 0.        ,
        2.5492556 ],
       [0.33036557, 1.3501908 , 1.3015912 , ..., 1.3371106 , 0.        ,
        2.515247  ],
       ...,
       [0.51509565, 0.80132717, 0.        , ..., 0.5189064 , 0.        ,
        0.        ],
       [0.4683566 , 0.12107759, 0.        , ..., 0.7368431 , 0.        ,
        0.        ],
       [0.6956396 , 0.        , 0.        , ..., 0.74605256, 0.        ,
        0.        ]], shape=(430800, 16), dtype=float32)

In [10]:
latent_scaler = StandardScaler()

X_train_latent_scaled = latent_scaler.fit_transform(X_train_latent)




In [11]:
ocsvm_latent = OneClassSVM(
    kernel="rbf",
    nu=0.1,
    gamma="scale"
    
)

In [12]:

#Get start time to find trainingtime
start_time = time.time()

ocsvm_latent.fit(X_train_latent_scaled)

#Get end time and calculate training time
end_time = time.time()
training_time = end_time - start_time

print(f"Training time: {training_time:.2f} seconds")
print(f"Training time: {training_time / 60:.2f} minutes")

Training time: 3175.96 seconds
Training time: 52.93 minutes


In [13]:
#Save model

# Lag mappe
model_dir = Path("models/saved_hybrid")
model_dir.mkdir(exist_ok=True)

# Lagre OCSVM-modellen
joblib.dump(ocsvm_latent, model_dir / "hybrid_model.pkl")

# Lagre scaler
joblib.dump(latent_scaler, model_dir / "scaler.pkl")

# Lagre feature-kolonner
joblib.dump(feature_columns, model_dir / "feature_columns.pkl")

print("hybrid-modell, scaler og feature columns er lagret.")

hybrid-modell, scaler og feature columns er lagret.


In [14]:
training_time

3175.9631867408752